In [0]:
%python
df = spark.readStream.format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

df.writeStream.format("console").start()

In [0]:
use catalog project

In [0]:
CREATE or replace TABLE  silver.sales_delta
(
  sale_id INT,
  product STRING,
  quantity INT,
  price DOUBLE,
  sale_timestamp TIMESTAMP
)
USING DELTA;


In [0]:
select * from silver.sales_delta;

sale_id,product,quantity,price,sale_timestamp
101,Laptop,1,55000.0,2025-01-10T10:00:00Z
102,Mouse,2,800.0,2025-01-10T10:05:00Z
103,Keyboard,1,1500.0,2025-01-10T10:15:00Z
108,Monitor,3,9500.0,2025-01-10T11:00:00Z
109,Charger,5,2200.0,2025-01-10T11:10:00Z
104,Monitor,1,8500.0,2025-01-10T11:00:00Z
105,Charger,3,1200.0,2025-01-10T11:10:00Z
104,Monitor,1,8500.0,2025-01-10T11:00:00Z
105,Charger,3,1200.0,2025-01-10T11:10:00Z
106,Monitor,1,8500.0,2025-01-10T11:00:00Z


In [0]:
%python
spark.sparkContext.setLogLevel("ERROR")
s="sale_id int,product string,quantity int,price int,sale_timestamp timestamp"
stream_df = spark.readStream.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load("abfss://data@trainingbr.dfs.core.windows.net/sales/",schema=s)

query = stream_df.writeStream \
        .format("delta") \
        .option("checkpointLocation", "abfss://data@trainingbr.dfs.core.windows.net/checkpoints/sales") \
        .outputMode("append") \
        .trigger(processingTime="30 seconds") \
        .table("project.silver.sales_delta")



In [0]:
select * from silver.sales_delta;

sale_id,product,quantity,price,sale_timestamp
101,Laptop,1,55000.0,2025-01-10T10:00:00Z
102,Mouse,2,800.0,2025-01-10T10:05:00Z
103,Keyboard,1,1500.0,2025-01-10T10:15:00Z
104,Monitor,1,8500.0,2025-01-10T11:00:00Z
105,Charger,3,1200.0,2025-01-10T11:10:00Z
104,Monitor,1,8500.0,2025-01-10T11:00:00Z
105,Charger,3,1200.0,2025-01-10T11:10:00Z
104,Monitor,1,8500.0,2025-01-10T11:00:00Z
105,Charger,3,1200.0,2025-01-10T11:10:00Z
104,Monitor,1,8500.0,2025-01-10T11:00:00Z


In [0]:
select * from silver.sales_delta

sale_id,product,quantity,price,sale_timestamp
101,Laptop,1,55000.0,2025-01-10T10:00:00Z
102,Mouse,2,800.0,2025-01-10T10:05:00Z
103,Keyboard,1,1500.0,2025-01-10T10:15:00Z
104,Monitor,1,8500.0,2025-01-10T11:00:00Z
105,Charger,3,1200.0,2025-01-10T11:10:00Z
104,Monitor,1,8500.0,2025-01-10T11:00:00Z
105,Charger,3,1200.0,2025-01-10T11:10:00Z
104,Monitor,1,8500.0,2025-01-10T11:00:00Z
105,Charger,3,1200.0,2025-01-10T11:10:00Z


In [0]:
-- delta table use for streaming

In [0]:
%python

from pyspark.sql.functions import to_date, sum ,col

# Incremental streaming read from Delta table
src_df = (
    spark.readStream
         .format("delta")
         .table("project.silver.sales_delta")
)

# Simple incremental aggregation: total amount per day
agg_df = (
    src_df
      .groupBy('product')
      .agg(sum(col("price") * col("quantity")).alias("total_sales"))
)

query = (
    agg_df.writeStream
          .format("delta")
          .outputMode("complete")  # we re-write the full aggregated result each trigger
          .option("checkpointLocation", "abfss://data@trainingbr.dfs.core.windows.net/checkpoints/daily_agg/")
          .table("project.silver.product_agg_sales")
)


In [0]:
%python

from pyspark.sql.functions import to_date, sum ,col

# Incremental streaming read from Delta table
src_df = (
    spark.readStream
         .format("delta")
         .table("project.silver.sales_delta")
)

# Simple incremental aggregation: total amount per day



query = (
    src_df.writeStream
          .format("delta")
          .outputMode("append") 
          .option("checkpointLocation", "abfss://data@trainingbr.dfs.core.windows.net/checkpoints/sales_append/")
          .table("project.silver.sales_delta_back")
)



In [0]:
%python

from pyspark.sql.functions import to_date, sum ,col

# Incremental streaming read from Delta table
src_df = (
    spark.readStream
         .format("delta")
         .table("project.silver.sales_delta")
)

src_df = src_df.filter(col("quantity") > 1)



query = (
    src_df.writeStream
          .format("delta")
          .outputMode("append") 
          .option("checkpointLocation", "abfss://data@trainingbr.dfs.core.windows.net/checkpoints/sales_append1/")
          .table("project.silver.sales_delta_back")
)



In [0]:
%python

from pyspark.sql.functions import to_date, sum ,col

# Incremental streaming read from Delta table
src_df = (
    spark.readStream
         .format("delta")
         .table("project.silver.sales_delta")
)

src_df = src_df.filter(col("quantity") > 1)



query = (
    src_df.writeStream
          .format("csv").option("path", "abfss://data@trainingbr.dfs.core.windows.net/sales_append_csv/")
          .outputMode("append") 
          .option("checkpointLocation", "abfss://data@trainingbr.dfs.core.windows.net/checkpoints/sales_append_csv/").start())
          




In [0]:
%python
df = spark.read.csv("abfss://data@trainingbr.dfs.core.windows.net/sales_append_csv/", header=True, inferSchema=True)
display(df)

108,Monitor,3,9500.0,2025-01-10T11:00:00.000Z
109,Charger,5,2200.0,2025-01-10T11:10:00Z
105,Charger,3,1200.0,2025-01-10T11:10:00Z
1002,Ram,2,200.0,2022-01-02T00:00:00Z
1003,Mouse Pad,3,300.0,2022-01-03T00:00:00Z
100,product1,10,100.0,2022-01-01T00:00:00Z
101,product2,20,200.0,2022-01-02T00:00:00Z
102,product3,30,300.0,2022-01-03T00:00:00Z
101,product2,20,200.0,2022-01-02T00:00:00Z
102,product3,30,300.0,2022-01-03T00:00:00Z
105,Charger,3,1200.0,2025-01-10T11:10:00Z


In [0]:
select * from  project.silver.sales_delta_back;

sale_id,product,quantity,price,sale_timestamp
2,Laptop,2,2000.0,2025-12-11T00:00:00Z
2,Laptop,2,2000.0,2025-12-11T00:00:00Z
102,Mouse,2,800.0,2025-01-10T10:05:00Z
108,Monitor,3,9500.0,2025-01-10T11:00:00Z
109,Charger,5,2200.0,2025-01-10T11:10:00Z
105,Charger,3,1200.0,2025-01-10T11:10:00Z
100,product1,10,100.0,2022-01-01T00:00:00Z
101,product2,20,200.0,2022-01-02T00:00:00Z
102,product3,30,300.0,2022-01-03T00:00:00Z
100,product1,10,100.0,2022-01-01T00:00:00Z


In [0]:
select * from project.silver.sales_delta_back;

sale_id,product,quantity,price,sale_timestamp
1,Monitor,1,2000.0,2025-12-11T00:00:00Z
2,Laptop,2,2000.0,2025-12-11T00:00:00Z
101,Laptop,1,55000.0,2025-01-10T10:00:00Z
102,Mouse,2,800.0,2025-01-10T10:05:00Z
103,Keyboard,1,1500.0,2025-01-10T10:15:00Z
108,Monitor,3,9500.0,2025-01-10T11:00:00Z
109,Charger,5,2200.0,2025-01-10T11:10:00Z
104,Monitor,1,8500.0,2025-01-10T11:00:00Z
105,Charger,3,1200.0,2025-01-10T11:10:00Z
100,product1,10,100.0,2022-01-01T00:00:00Z


In [0]:
select * from silver.product_agg_sales;

product,total_sales
product3,18000.0
product2,8000.0
Keyboard,1500.0
product1,2000.0
Charger,21800.0
Monitor,56000.0
Laptop,59000.0
Mouse,1600.0


In [0]:
insert into silver.sales_delta values( 100000,'Monitor',1,2000,current_date()),
                                     (2,'Laptop',2,2000,current_date())

num_affected_rows,num_inserted_rows
2,2


In [0]:
insert into silver.sales_delta values(1001,'joystic',1,100,'2022-01-01'),
                                      (1002,'Ram',2,200,'2022-01-02'),
                                      (1003,'Mouse Pad',3,300,'2022-01-03')
                                      

num_affected_rows,num_inserted_rows
3,3


num_affected_rows
2


In [0]:
%python
spark.sparkContext.setLogLevel("ERROR")

s = "sale_id int,product string,quantity int,price int,sale_timestamp timestamp"

stream_df = (
    spark.readStream.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .schema(s)
    .load("abfss://data@trainingbr.dfs.core.windows.net/sales/")
)

query = (
    stream_df.writeStream
    .format("console")
    .outputMode("update")
    .trigger(processingTime="30 seconds")
    .start()
)

In [0]:
select * from temp_view;

sale_id,product,quantity,price,sale_timestamp
101,Laptop,1,55000,2025-01-10T10:00:00Z
102,Mouse,2,800,2025-01-10T10:05:00Z
103,Keyboard,1,1500,2025-01-10T10:15:00Z


In [0]:
select * from temp_view;

sale_id,product,quantity,price,sale_timestamp
101,Laptop,1,55000,2025-01-10T10:00:00Z
102,Mouse,2,800,2025-01-10T10:05:00Z
103,Keyboard,1,1500,2025-01-10T10:15:00Z
101,Laptop,2,80000,2025-01-10T10:00:00Z
102,Mouse,3,900,2025-01-10T10:05:00Z
103,Keyboard,1,2000,2025-01-10T10:15:00Z


In [0]:
truncate table silver.sales_delta;